# RQ2 — Prompt Injection Walkthrough

**Purpose:** This notebook provides a step-by-step walkthrough of how *ESRS context injection*
(referred to as "prompt injection" in the thesis) is implemented in the RQ2 evaluation pipeline.
It shows exactly what the LLM sees in each experimental condition and demonstrates the difference
across multiple concrete examples spanning different ESRS standards and companies.

**Research Question 2:** *To what extent does full-standard ESRS context injection improve
evidence-grounded compliance assessment?*

**Experimental conditions:**

| Condition | Label | What the LLM receives |
|-----------|-------|----------------------|
| **full_system** | With ESRS context | System prompt + Question + **ESRS requirements** + Retrieved evidence |
| **no_esrs_context** | Without ESRS context | System prompt + Question + *(empty ESRS block)* + Retrieved evidence |

The only variable that changes between conditions is whether the ESRS regulatory standard
text is injected into the prompt. Everything else — the retrieval pipeline, the system prompt,
the company report evidence, and the output schema — remains identical.

## 1. Setup and Imports

In [1]:
import sys, os, json, textwrap, hashlib, html as html_mod, importlib.util
from pathlib import Path
from IPython.display import display, Markdown, HTML

# Ensure the project root is importable
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rag.config.settings import (
    GOLDEN_STANDARD_PATH, ESRS_JSON_DIR, RQ2_RESULTS_DIR, BEST_PIPELINE_PATH,
)
from rag.data.loaders import load_generation_samples
from rag.types import GenerationSample, RetrievedEvidenceChunk

# Import ESRS context and prompt modules directly via importlib to avoid triggering
# the generation __init__.py which pulls in the OpenAI SDK (not needed for cached analysis)
def _import_module(name, path):
    spec = importlib.util.spec_from_file_location(name, PROJECT_ROOT / path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_esrs_mod = _import_module("rag.generation.esrs_context", "rag/generation/esrs_context.py")
_prompts_mod = _import_module("rag.generation.prompts", "rag/generation/prompts.py")

load_esrs_standards = _esrs_mod.load_esrs_standards
extract_esrs_context = _esrs_mod.extract_esrs_context
build_disclosure_context = _esrs_mod.build_disclosure_context
build_answer_messages = _prompts_mod.build_answer_messages
format_retrieved_chunks = _prompts_mod.format_retrieved_chunks

# Cache-only answer loading — avoids requiring the OpenAI SDK at import time
def _cache_key(question, company, condition, esrs_context):
    blob = f"{question}|{company}|{condition}|{len(esrs_context)}"
    return hashlib.sha256(blob.encode()).hexdigest()[:16]

def load_cached_answer(*, sample, esrs_context, condition):
    """Load a previously cached generation result without calling the API."""
    cache_dir = RQ2_RESULTS_DIR / "nb_cache" / condition
    key = _cache_key(sample.question, sample.company, condition, esrs_context)
    cache_file = cache_dir / f"{key}.json"
    if cache_file.exists():
        return json.loads(cache_file.read_text(encoding="utf-8"))
    raise FileNotFoundError(f"No cached result for {sample.company} × {condition} ({cache_file.name})")

print(f"Project root : {PROJECT_ROOT}")
print(f"ESRS JSON dir: {ESRS_JSON_DIR}")
print(f"RQ2 cache dir: {RQ2_RESULTS_DIR / 'nb_cache'}")

Project root : /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot
ESRS JSON dir: /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/data/ESRS_Standards_JSON
RQ2 cache dir: /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/data/artifacts/evaluation_results/rq2/nb_cache


## 2. Architecture Overview — Where Prompt Injection Happens

The RQ2 pipeline follows a standard RAG (Retrieval-Augmented Generation) pattern with one
experimental twist: the **ESRS regulatory context** can be injected or withheld.

```
┌──────────────────────────────────────────────────────────────────────┐
│                        RQ2 Generation Pipeline                       │
├──────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  ┌─────────────┐    ┌──────────────────┐    ┌────────────────────┐  │
│  │ Golden       │───▶│ Best RQ1         │───▶│ Retrieved Evidence │  │
│  │ Standard     │    │ Retrieval        │    │ Chunks (top-15)    │  │
│  │ (question +  │    │ Pipeline         │    │                    │  │
│  │  company)    │    │                  │    │ (identical for     │  │
│  └─────────────┘    └──────────────────┘    │  both conditions)  │  │
│                                              └─────────┬──────────┘  │
│                                                        │             │
│  ┌─────────────────────────────────────┐               │             │
│  │ ESRS Standards JSON                 │               │             │
│  │ (e.g. ESRS_E1.json)                │               │             │
│  │                                     │               │             │
│  │ ┌─ full_system ──────────────┐      │               │             │
│  │ │  Flatten & serialize full  │──────┼───┐           │             │
│  │ │  regulatory text           │      │   │           │             │
│  │ └───────────────────────────┘      │   │           │             │
│  │ ┌─ no_esrs_context ─────────┐      │   │           │             │
│  │ │  Empty string ""          │──────┼───┤           │             │
│  │ └───────────────────────────┘      │   │           │             │
│  └─────────────────────────────────────┘   │           │             │
│                                            ▼           ▼             │
│                              ┌──────────────────────────────┐       │
│                              │      build_answer_messages()  │       │
│                              │  ┌──────────────────────────┐ │       │
│                              │  │ SYSTEM: evaluation rubric │ │       │
│                              │  │ USER:   question          │ │       │
│                              │  │       + ESRS context ◄────┼─┼─ THE INJECTION POINT
│                              │  │       + evidence chunks   │ │       │
│                              │  └──────────────────────────┘ │       │
│                              └──────────────┬───────────────┘       │
│                                             │                        │
│                                             ▼                        │
│                              ┌──────────────────────────────┐       │
│                              │  OpenAI API (gpt-4o-mini)     │       │
│                              │  + JSON schema enforcement    │       │
│                              └──────────────┬───────────────┘       │
│                                             │                        │
│                                             ▼                        │
│                              ┌──────────────────────────────┐       │
│                              │  Structured JSON response:    │       │
│                              │  • compliance_level           │       │
│                              │  • compliance_rationale       │       │
│                              │  • key_findings               │       │
│                              │  • disclosure_gaps            │       │
│                              │  • cited_pages                │       │
│                              └──────────────────────────────┘       │
└──────────────────────────────────────────────────────────────────────┘
```

The key source files involved:

| File | Role |
|------|------|
| `rag/generation/esrs_context.py` | Loads and serializes ESRS JSON into a prompt-ready text block |
| `rag/generation/prompts.py` | Assembles the full system + user message with the injection point |
| `rag/generation/generator.py` | Calls the OpenAI API with strict JSON schema enforcement |

## 3. Concrete Examples — Multi-Standard, Multi-Company

To illustrate how the pipeline works across different ESRS domains, we examine **four questions**
spanning environmental, social, and governance standards, each with **three companies** that
cover a range of fulfillment levels:

| Question | ESRS Standard | Disclosure IDs | Domain |
|----------|--------------|----------------|--------|
| Q3 | **ESRS E1** — Climate Change | E1-1 | Environmental (Transition Plan) |
| Q9 | **ESRS S2** — Workers in the Value Chain | S2-1, S2-2 | Social |
| Q12 | **ESRS G1** — Business Conduct | G1-3, G1-4 | Governance |
| Q1 | **ESRS 2** — General Disclosures | IRO-1, SBM-3 | Cross-cutting |

**Selected companies:** Aker BP, Galp, Neste

### 3.1 Load and Compare Gold-Standard Samples

In [2]:
# Load all 204 generation samples
all_samples = load_generation_samples()
print(f"Loaded {len(all_samples)} generation samples")

# Define our example set: 4 questions × 3 companies
EXAMPLE_QUESTIONS = ["Q3", "Q9", "Q12", "Q1"]
EXAMPLE_COMPANIES = ["Aker BP", "Galp", "Neste"]

# Build a lookup: (question_id, company) → sample
sample_lookup = {(s.question_id, s.company): s for s in all_samples}

# Collect our example samples
examples = {}
for qid in EXAMPLE_QUESTIONS:
    examples[qid] = {}
    for comp in EXAMPLE_COMPANIES:
        examples[qid][comp] = sample_lookup[(qid, comp)]

# Display one summary table per question
for qid in EXAMPLE_QUESTIONS:
    first = examples[qid][EXAMPLE_COMPANIES[0]]

    # Build HTML table
    tbl = f'''<h4 style="margin-top:1.5em;">{qid} — {first.esrs_standard} (Disclosure: {first.esrs_disclosure_id})</h4>
<p style="background:#2c3e50; color:white; font-size:0.9em; margin-bottom:0.5em; padding:8px 12px; border-radius:6px;"><em style="color:white;">{html_mod.escape(first.question[:150])}{"..." if len(first.question)>150 else ""}</em></p>
<table style="border-collapse:collapse; width:100%; font-size:0.9em;">
<tr style="background:#f0f4f8;">
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">Attribute</th>'''

    for comp in EXAMPLE_COMPANIES:
        tbl += f'<th style="border:1px solid #ccc; padding:8px; text-align:center;">{comp}</th>'
    tbl += '</tr>'

    # Fulfillment level row with color coding
    tbl += '<tr><td style="border:1px solid #ccc; padding:8px; font-weight:bold;">Fulfillment Level</td>'
    for comp in EXAMPLE_COMPANIES:
        s = examples[qid][comp]
        color = {"High": "#27ae60", "Medium": "#f39c12", "Low": "#e74c3c"}.get(s.fulfillment_level, "#333")
        tbl += f'<td style="border:1px solid #ccc; padding:8px; text-align:center; color:{color}; font-weight:bold; font-size:1.1em;">{s.fulfillment_level}</td>'
    tbl += '</tr>'

    # Number of evidence passages
    tbl += '<tr style="background:#fafafa;"><td style="border:1px solid #ccc; padding:8px;">Evidence Passages</td>'
    for comp in EXAMPLE_COMPANIES:
        s = examples[qid][comp]
        tbl += f'<td style="border:1px solid #ccc; padding:8px; text-align:center;">{len(s.evidences)}</td>'
    tbl += '</tr></table>'

    display(HTML(tbl))

Loaded 204 generation samples


Attribute,Aker BP,Galp,Neste
Fulfillment Level,High,Medium,Medium
Evidence Passages,3,4,3


Attribute,Aker BP,Galp,Neste
Fulfillment Level,High,High,High
Evidence Passages,3,3,3


Attribute,Aker BP,Galp,Neste
Fulfillment Level,High,High,Medium
Evidence Passages,4,3,3


Attribute,Aker BP,Galp,Neste
Fulfillment Level,High,High,High
Evidence Passages,3,3,3


### 3.2 Gold-Standard Fulfillment Assessments

Each sample in the golden standard includes a **fulfillment level** (High / Medium / Low) and
a **fulfillment rationale** that explains the expert's reasoning. These serve as the ground
truth against which the LLM's predictions are evaluated.

In [3]:
# Display fulfillment level and rationale for each question × company as styled HTML cards
for qid in EXAMPLE_QUESTIONS:
    first = examples[qid][EXAMPLE_COMPANIES[0]]
    display(HTML(f'<h4 style="margin-top:2em; border-bottom:2px solid #3498db; padding-bottom:5px;">{qid} — {first.esrs_standard}</h4>'))

    cards_html = '<div style="display:flex; gap:12px; flex-wrap:wrap;">'
    for comp in EXAMPLE_COMPANIES:
        s = examples[qid][comp]
        level = s.fulfillment_level
        color = {"High": "#27ae60", "Medium": "#f39c12", "Low": "#e74c3c"}.get(level, "#333")
        bg = {"High": "#eafaf1", "Medium": "#fef9e7", "Low": "#fdedec"}.get(level, "#f9f9f9")
        border = {"High": "#27ae60", "Medium": "#f39c12", "Low": "#e74c3c"}.get(level, "#ccc")

        rationale = s.fulfillment_rationale or "No rationale provided."
        rationale_display = html_mod.escape(rationale[:400]) + ("…" if len(rationale) > 400 else "")

        cards_html += f'''
        <div style="flex:1; min-width:280px; border:2px solid {border}; border-radius:8px;
                     padding:12px; background:{bg};">
            <div style="font-size:1.1em; font-weight:bold; margin-bottom:4px;">{html_mod.escape(comp)}</div>
            <div style="font-size:1.3em; font-weight:bold; color:{color}; margin-bottom:8px;">
                {"●" * (3 if level=="High" else 2 if level=="Medium" else 1)} {level}
            </div>
            <div style="font-size:0.85em; color:#333; line-height:1.4;">
                {rationale_display}
            </div>
        </div>'''
    cards_html += '</div>'
    display(HTML(cards_html))

## 4. Component 1 — The ESRS Context (The "Injection")

This is the core of the experiment. Each ESRS standard's text is loaded from a structured JSON
file (e.g., `data/ESRS_Standards_JSON/ESRS_E1.json`), flattened into a hierarchical text
representation, and filtered to the relevant disclosure IDs.

The function `build_disclosure_context()` in `rag/generation/esrs_context.py` handles this:
1. It parses the nested JSON tree of the ESRS standard
2. It filters to subtrees matching the disclosure IDs (e.g., E1-1)
3. It serializes each paragraph as `[Paragraph X / Section > Heading] text`

This text block is what gets **injected** into the prompt in the `full_system` condition.
In the `no_esrs_context` condition, this is replaced with an empty string.

Below we show the injected context for each of our example standards, grouped by ESRS domain.

In [4]:
# Load all ESRS standard JSON files
standards = load_esrs_standards()
print(f"Loaded {len(standards)} ESRS standards: {list(standards.keys())}")

# Extract and display ESRS context for each unique standard in our examples
esrs_contexts = {}  # qid → context string

# Group by ESRS domain for organized display
standard_groups = {
    "Environmental": ["Q3"],
    "Social": ["Q9"],
    "Governance": ["Q12"],
    "Cross-cutting": ["Q1"],
}

for group_name, qids in standard_groups.items():
    display(HTML(f'<h4 style="margin-top:1.5em; color:#2c3e50;">{group_name} Standards</h4>'))

    for qid in qids:
        s = examples[qid][EXAMPLE_COMPANIES[0]]  # context is same for all companies

        ctx = extract_esrs_context(
            esrs_standard=s.esrs_standard,
            standards=standards,
            question=s.question,
            esrs_disclosure_id=s.esrs_disclosure_id,
        )
        esrs_contexts[qid] = ctx

        ctx_escaped = html_mod.escape(ctx[:2000])

        display(HTML(f'''
        <div style="margin:0.5em 0 1em 0; padding:12px; border-left:4px solid #3498db; background:#f8f9fa;">
            <h4 style="margin:0 0 5px 0;">{s.esrs_standard} — Disclosure {s.esrs_disclosure_id}</h4>
            <p style="margin:0 0 8px 0; color:#666; font-size:0.9em;">Context length: {len(ctx):,} characters</p>
            <details>
                <summary style="cursor:pointer; color:#3498db; font-weight:bold;">Click to expand ESRS context (first 2,000 chars)</summary>
                <pre style="white-space:pre-wrap; font-size:0.8em; max-height:300px; overflow-y:auto;
                            background:#fff; padding:8px; border:1px solid #ddd; margin-top:8px;">{ctx_escaped}{"..." if len(ctx)>2000 else ""}</pre>
            </details>
        </div>
        '''))

Loaded 12 ESRS standards: ['ESRS 1', 'ESRS 2', 'ESRS E1', 'ESRS E2', 'ESRS E3', 'ESRS E4', 'ESRS E5', 'ESRS G1', 'ESRS S1', 'ESRS S2', 'ESRS S3', 'ESRS S4']


### What Each Condition Sees

| | `full_system` | `no_esrs_context` |
|---|---|---|
| **ESRS context** | ~14,000–25,000 chars of regulatory text | `""` (empty string) |
| **System Prompt** | Default System Prompt + explaination on what the ESRS Requirements block is and how it should integrate it | Default System Prompt |
| **Everything else** | Identical | Identical |

The hypothesis is that injecting this regulatory text helps the LLM better interpret what
constitutes adequate disclosure fulfillment. The experiment tests whether this actually improves
compliance assessment accuracy and rationale quality for the golden standard document or not.

## 5. Component 2 — The System Prompt

The system prompt defines the LLM's role and the evaluation rubric. It is **identical** across
both conditions (WITH THE EXCEPTION OF THE ESRS REQUIREMENTS FOR THE FULL SYSTEM) — this is critical for experimental control. The same system prompt is used
regardless of the ESRS standard being evaluated.

Key elements of the system prompt:
- **Role definition:** The LLM evaluates ESRS disclosure fulfillment (not sustainability performance quality)
- **Compliance rubric:** High = evidence clearly and substantially answers the question; Medium = partial answer or material ambiguity; Low = missing or only superficial answer
- **Grounding constraint:** All claims must be grounded in retrieved evidence
- **ESRS context instruction:** Differs by condition — `full_system` is told to use ESRS context as interpretive reference; `no_esrs_context` is told not to invent requirements from memory
- **Output schema:** Specifies the exact JSON keys: `direct_answer`, `compliance_level`, `compliance_rationale`, `key_findings`, `disclosure_gaps`, `cited_pages`

In [5]:
# Build mock evidence chunks from one example to demonstrate prompt construction
sample_demo = examples["Q3"]["Aker BP"]
mock_chunks = [
    RetrievedEvidenceChunk(
        chunk_id=f"gold_{i}",
        text=ev.text,
        score=1.0,
        rank=i + 1,
        page_start=int(ev.page_number) if ev.page_number.isdigit() else None,
        page_end=int(ev.page_number) if ev.page_number.isdigit() else None,
        header_hierarchy=[ev.source_section_header] if ev.source_section_header else [],
    )
    for i, ev in enumerate(sample_demo.evidences)
]

# Build the full prompt messages for the full_system condition
messages_with_context = build_answer_messages(
    sample=sample_demo,
    retrieved_chunks=mock_chunks,
    esrs_context=esrs_contexts["Q3"],
)

system_escaped = html_mod.escape(messages_with_context[0]["content"])

display(HTML(f'''<h4>System Prompt (shared by both conditions)</h4>
<pre style="white-space:pre-wrap; background:#f4f4f4; padding:12px; border:1px solid #ddd;
            font-size:0.85em; max-height:400px; overflow-y:auto;">{system_escaped}</pre>'''))

## 6. Component 3 — The User Message (Side-by-Side Comparison)

This is where the injection happens. The user message is constructed by `build_answer_messages()`
in `rag/generation/prompts.py`. Its structure is:

```
Condition: {mode_label}
Company: {company}
ESRS standard: {esrs_standard}
Disclosure ID: {disclosure_id}
Question: {question}

ESRS requirements context (regulatory reference, not company evidence):
{esrs_context}              ← THIS IS THE INJECTION POINT

Retrieved company report evidence:
{evidence_block}

Write a faithful direct answer, assign the disclosure fulfillment level, ...
```

When the ESRS context is empty (in the `no_esrs_context` condition), the requirements section
is simply omitted — no placeholder text or empty block is shown.

In [6]:
# Build messages for BOTH conditions using the ESRS E1 / Aker BP example
messages_full = build_answer_messages(
    sample=sample_demo, retrieved_chunks=mock_chunks, esrs_context=esrs_contexts["Q3"],
)
messages_no_ctx = build_answer_messages(
    sample=sample_demo, retrieved_chunks=mock_chunks, esrs_context="",
)

user_full = messages_full[1]["content"]
user_no_ctx = messages_no_ctx[1]["content"]

print(f"User message length WITH ESRS context   : {len(user_full):>8,} chars")
print(f"User message length WITHOUT ESRS context : {len(user_no_ctx):>8,} chars")
print(f"Difference (= ESRS requirements text)    : {len(user_full) - len(user_no_ctx):>8,} chars")

User message length WITH ESRS context   :   12,141 chars
User message length WITHOUT ESRS context :    1,475 chars
Difference (= ESRS requirements text)    :   10,666 chars


### 6.1 User Message — `full_system` (WITH ESRS Context)

In [7]:
# Show the full user message with ESRS context
lines = user_full.split("\n")

html_out = '<div style="background:#f0f7ff; border:2px solid #3498db; border-radius:8px; padding:16px;">'
html_out += '<h4 style="color:#2c3e50; margin-top:0;">USER MESSAGE — full_system condition</h4>'

# Question metadata section
meta_lines = [l for l in lines[:6] if l.strip()]
html_out += f'<pre style="background:#fff; padding:10px; border:1px solid #ddd; font-size:0.85em;">{html_mod.escape(chr(10).join(meta_lines))}</pre>'

# ESRS requirements section
esrs_marker = "ESRS requirements context (regulatory reference, not company evidence):"
esrs_start = user_full.find(esrs_marker)
if esrs_start >= 0:
    esrs_start += len(esrs_marker)
    evidence_marker = "Retrieved company report evidence"
    esrs_end = user_full.find(evidence_marker)
    esrs_block = user_full[esrs_start:esrs_end].strip()
    esrs_lines = esrs_block.split("\n")
    preview = chr(10).join(esrs_lines[:15])
    more = f"\n... [{len(esrs_lines) - 15} more lines]" if len(esrs_lines) > 15 else ""
    html_out += f'''
    <div style="margin:8px 0; padding:10px; border-left:4px solid #e67e22; background:#fef9e7;">
        <strong style="color:#e67e22;">ESRS Requirements Context</strong>
        <span style="color:#888;"> ({len(esrs_lines)} lines, {len(esrs_block):,} chars)</span>
        <pre style="font-size:0.8em; max-height:200px; overflow-y:auto; margin-top:6px;">{html_mod.escape(preview + more)}</pre>
    </div>'''

# Evidence section
evidence_start = user_full.find("Retrieved company report evidence")
if evidence_start >= 0:
    evidence_block = user_full[evidence_start:]
    ev_lines = evidence_block.strip().split("\n")
    preview_ev = chr(10).join(ev_lines[:12])
    more_ev = f"\n... [{len(ev_lines) - 12} more lines]" if len(ev_lines) > 12 else ""
    html_out += f'''
    <div style="margin:8px 0; padding:10px; border-left:4px solid #27ae60; background:#eafaf1;">
        <strong style="color:#27ae60;">Retrieved Company Evidence</strong>
        <pre style="font-size:0.8em; max-height:200px; overflow-y:auto; margin-top:6px;">{html_mod.escape(preview_ev + more_ev)}</pre>
    </div>'''

html_out += '</div>'
display(HTML(html_out))

### 6.2 User Message — `no_esrs_context` (WITHOUT ESRS Context)

In [8]:
# Show the user message WITHOUT ESRS context
lines_no_ctx = user_no_ctx.split("\n")

html_out = '<div style="background:#fdf2f2; border:2px solid #e74c3c; border-radius:8px; padding:16px;">'
html_out += '<h4 style="color:#2c3e50; margin-top:0;">USER MESSAGE — no_esrs_context condition</h4>'

# Question metadata section
meta_lines_no = [l for l in lines_no_ctx[:6] if l.strip()]
html_out += f'<pre style="background:#fff; padding:10px; border:1px solid #ddd; font-size:0.85em;">{html_mod.escape(chr(10).join(meta_lines_no))}</pre>'

# Show that ESRS context is absent
esrs_marker = "ESRS requirements context"
if esrs_marker not in user_no_ctx:
    html_out += '''
    <div style="margin:8px 0; padding:10px; border-left:4px solid #95a5a6; background:#f5f5f5;">
        <strong style="color:#95a5a6;">ESRS Requirements Context</strong>
        <span style="color:#999; font-style:italic;"> — Not present in this condition</span>
    </div>'''

# Evidence section (same as full_system)
evidence_start = user_no_ctx.find("Retrieved company report evidence")
if evidence_start >= 0:
    evidence_block_no = user_no_ctx[evidence_start:]
    ev_lines_no = evidence_block_no.strip().split("\n")
    preview_no = chr(10).join(ev_lines_no[:12])
    more_no = f"\n... [{len(ev_lines_no) - 12} more lines]" if len(ev_lines_no) > 12 else ""
    html_out += f'''
    <div style="margin:8px 0; padding:10px; border-left:4px solid #27ae60; background:#eafaf1;">
        <strong style="color:#27ae60;">Retrieved Company Evidence</strong>
        <span style="color:#888;"> (identical to full_system)</span>
        <pre style="font-size:0.8em; max-height:200px; overflow-y:auto; margin-top:6px;">{html_mod.escape(preview_no + more_no)}</pre>
    </div>'''

html_out += '</div>'
display(HTML(html_out))

## 7. Component 4 — The JSON Schema Enforcement

The OpenAI API call uses **strict JSON schema mode** to constrain the model's output to a
well-defined structure. This ensures that both conditions produce responses in the same format,
making them directly comparable.

The schema is defined in `rag/generation/generator.py` and enforced via the OpenAI Responses API's
`text.format.json_schema` parameter with `strict: True`.

In [9]:
# The JSON schema enforced on the LLM output
output_schema = {
    "type": "object",
    "properties": {
        "direct_answer": {"type": "string", "minLength": 1},
        "compliance_level": {"type": "string", "enum": ["Low", "Medium", "High"]},
        "compliance_rationale": {"type": "string", "minLength": 1},
        "key_findings": {"type": "array", "items": {"type": "string"}},
        "disclosure_gaps": {"type": "array", "items": {"type": "string"}},
        "cited_pages": {"type": "array", "items": {"type": "string"}},
    },
    "required": [
        "direct_answer", "compliance_level", "compliance_rationale",
        "key_findings", "disclosure_gaps", "cited_pages",
    ],
    "additionalProperties": False,
}

print("Enforced JSON output schema:")
print(json.dumps(output_schema, indent=2))

print("\n" + "─" * 80)
print("API call configuration:")
print("─" * 80)
print(f"  Model              : gpt-4o-mini (via OPENAI_API_KEY)")
print(f"  Reasoning effort   : medium")
print(f"  Max output tokens  : 4096")
print(f"  Schema mode        : strict (guaranteed valid JSON)")
print(f"  Repair strategy    : If first response fails validation, a repair prompt is sent")

Enforced JSON output schema:
{
  "type": "object",
  "properties": {
    "direct_answer": {
      "type": "string",
      "minLength": 1
    },
    "compliance_level": {
      "type": "string",
      "enum": [
        "Low",
        "Medium",
        "High"
      ]
    },
    "compliance_rationale": {
      "type": "string",
      "minLength": 1
    },
    "key_findings": {
      "type": "array",
      "items": {
        "type": "string"
      }
    },
    "disclosure_gaps": {
      "type": "array",
      "items": {
        "type": "string"
      }
    },
    "cited_pages": {
      "type": "array",
      "items": {
        "type": "string"
      }
    }
  },
  "required": [
    "direct_answer",
    "compliance_level",
    "compliance_rationale",
    "key_findings",
    "disclosure_gaps",
    "cited_pages"
  ],
  "additionalProperties": false
}

────────────────────────────────────────────────────────────────────────────────
API call configuration:
──────────────────────────────────────

## 8. Live Execution — Running Both Conditions

We now load the generation results for all our example combinations. For each of the
4 questions and 3 companies (12 combinations total), we have cached results from both conditions:

| Step | Description |
|------|------------|
| 1 | Select sample (question × company) from the golden standard |
| 2 | Extract the gold-standard evidence passages as input |
| 3 | Build ESRS context from the relevant standard JSON |
| 4 | Call `generate_answer()` under `full_system` condition (ESRS context injected) |
| 5 | Call `generate_answer()` under `no_esrs_context` condition (empty ESRS block) |
| 6 | Compare both outputs against the gold-standard fulfillment level |

> **Note:** Results are loaded from the file-based cache (`nb_cache/`). Set `use_cache=False`
> in `generate_answer()` to force fresh API calls if needed.

In [10]:
# Load cached results for all 12 example combinations × 2 conditions
results = {}  # (qid, company, condition) → result dict

total = len(EXAMPLE_QUESTIONS) * len(EXAMPLE_COMPANIES) * 2
completed = 0
errors = []

display(HTML('<h4>Loading cached generation results</h4>'))

for qid in EXAMPLE_QUESTIONS:
    first = examples[qid][EXAMPLE_COMPANIES[0]]
    print(f"\n{qid} — {first.esrs_standard} ({first.esrs_disclosure_id})")
    print("─" * 70)

    for comp in EXAMPLE_COMPANIES:
        s = examples[qid][comp]
        ctx = esrs_contexts[qid]

        for condition, esrs_ctx in [("full_system", ctx), ("no_esrs_context", "")]:
            try:
                result = load_cached_answer(sample=s, esrs_context=esrs_ctx, condition=condition)
                results[(qid, comp, condition)] = result
                completed += 1
            except FileNotFoundError as e:
                errors.append(str(e))
                results[(qid, comp, condition)] = {"compliance_level": "?", "answer": "", "rationale": ""}
                completed += 1

        # Display per-company results
        r_full = results[(qid, comp, "full_system")]
        r_none = results[(qid, comp, "no_esrs_context")]
        gold = s.fulfillment_level

        full_icon = "✓" if r_full["compliance_level"] == gold else "✗"
        none_icon = "✓" if r_none["compliance_level"] == gold else "✗"

        print(f"  {comp:12} │ Gold: {gold:6} │ full_system: {r_full['compliance_level']:6} {full_icon} │ no_esrs_context: {r_none['compliance_level']:6} {none_icon}")

print(f"\n{'═' * 70}")
print(f"✓ Loaded {completed} results ({completed//2} samples × 2 conditions)")
if errors:
    print(f"⚠ {len(errors)} cache misses (results may be incomplete)")


Q3 — ESRS E1 (E1-1)
──────────────────────────────────────────────────────────────────────
  Aker BP      │ Gold: High   │ full_system: High   ✓ │ no_esrs_context: High   ✓
  Galp         │ Gold: Medium │ full_system: Medium ✓ │ no_esrs_context: Medium ✓
  Neste        │ Gold: Medium │ full_system: Medium ✓ │ no_esrs_context: High   ✗

Q9 — ESRS S2 (S2-1, S2-2)
──────────────────────────────────────────────────────────────────────
  Aker BP      │ Gold: High   │ full_system: High   ✓ │ no_esrs_context: High   ✓
  Galp         │ Gold: High   │ full_system: High   ✓ │ no_esrs_context: High   ✓
  Neste        │ Gold: High   │ full_system: High   ✓ │ no_esrs_context: High   ✓

Q12 — ESRS G1 (G1-3, G1-4)
──────────────────────────────────────────────────────────────────────
  Aker BP      │ Gold: High   │ full_system: High   ✓ │ no_esrs_context: High   ✓
  Galp         │ Gold: High   │ full_system: High   ✓ │ no_esrs_context: High   ✓
  Neste        │ Gold: Medium │ full_system: High   ✗ │

## 9. Results Comparison — Model Output vs. Golden Standard

For each question and company, we compare the LLM's predicted compliance level and rationale
against the expert-annotated golden standard. This lets us see where ESRS context injection
helps, where it makes no difference, and where discrepancies arise.

### 9.1 Compliance Level and Rationale — Detailed Comparison

In [11]:
# Build rich HTML comparison cards for each question × company
for qid in EXAMPLE_QUESTIONS:
    first = examples[qid][EXAMPLE_COMPANIES[0]]
    display(HTML(f'''
    <div style="margin:2em 0 0.5em 0; padding:10px 15px; background:#2c3e50; color:white;
                border-radius:8px 8px 0 0;">
        <h3 style="margin:0; color:white;">{qid} — {first.esrs_standard} ({first.esrs_disclosure_id})</h3>
        <p style="margin:4px 0 0 0; font-size:0.85em; opacity:0.9;">{html_mod.escape(first.question[:150])}...</p>
    </div>
    '''))

    for comp in EXAMPLE_COMPANIES:
        s = examples[qid][comp]
        r_full = results[(qid, comp, "full_system")]
        r_none = results[(qid, comp, "no_esrs_context")]
        gold = s.fulfillment_level

        def level_badge(level, is_match):
            color = {"High": "#27ae60", "Medium": "#f39c12", "Low": "#e74c3c"}.get(level, "#333")
            bg = {"High": "#eafaf1", "Medium": "#fef9e7", "Low": "#fdedec"}.get(level, "#f9f9f9")
            icon = "✓" if is_match else "✗"
            icon_color = "#27ae60" if is_match else "#e74c3c"
            return f'<span style="background:{bg}; color:{color}; padding:3px 10px; border-radius:12px; font-weight:bold; border:1px solid {color};">{level}</span> <span style="color:{icon_color}; font-weight:bold;">{icon}</span>'

        gold_badge = f'<span style="background:#eee; color:#333; padding:3px 10px; border-radius:12px; font-weight:bold; border:1px solid #999;">{gold}</span>'

        full_match = r_full["compliance_level"] == gold
        none_match = r_none["compliance_level"] == gold

        def truncate(text, max_len=500):
            if not text:
                return "<em style='color:#999'>No rationale provided</em>"
            t = html_mod.escape(text)
            if len(t) > max_len:
                return t[:max_len] + "…"
            return t

        gold_rationale = truncate(s.fulfillment_rationale)
        full_rationale = truncate(r_full.get("rationale", ""))
        none_rationale = truncate(r_none.get("rationale", ""))

        html_str = f'''
        <div style="border:1px solid #ddd; border-radius:0 0 4px 4px; margin-bottom:12px; overflow:hidden;">
            <div style="background:#ecf0f1; padding:8px 15px; font-weight:bold; font-size:1.05em;
                        border-bottom:1px solid #ddd;">
                {html_mod.escape(comp)}
            </div>
            <table style="width:100%; border-collapse:collapse; font-size:0.88em;">
                <tr style="background:#f8f9fa;">
                    <th style="padding:8px 12px; text-align:left; width:18%; border-bottom:1px solid #eee;"></th>
                    <th style="padding:8px 12px; text-align:center; width:27%; border-bottom:1px solid #eee; color:#2c3e50;">
                        Golden Standard
                    </th>
                    <th style="padding:8px 12px; text-align:center; width:27%; border-bottom:1px solid #eee; color:#3498db;">
                        full_system
                    </th>
                    <th style="padding:8px 12px; text-align:center; width:27%; border-bottom:1px solid #eee; color:#e74c3c;">
                        no_esrs_context
                    </th>
                </tr>
                <tr>
                    <td style="padding:8px 12px; font-weight:bold; border-right:1px solid #eee;">Level</td>
                    <td style="padding:8px 12px; text-align:center; border-right:1px solid #eee;">{gold_badge}</td>
                    <td style="padding:8px 12px; text-align:center; border-right:1px solid #eee;">{level_badge(r_full["compliance_level"], full_match)}</td>
                    <td style="padding:8px 12px; text-align:center;">{level_badge(r_none["compliance_level"], none_match)}</td>
                </tr>
                <tr style="background:#fafafa;">
                    <td style="padding:8px 12px; font-weight:bold; vertical-align:top; border-right:1px solid #eee;">Rationale</td>
                    <td style="padding:8px 12px; font-size:0.9em; line-height:1.4; border-right:1px solid #eee; color:#555;">
                        {gold_rationale}
                    </td>
                    <td style="padding:8px 12px; font-size:0.9em; line-height:1.4; border-right:1px solid #eee; color:#555;">
                        {full_rationale}
                    </td>
                    <td style="padding:8px 12px; font-size:0.9em; line-height:1.4; color:#555;">
                        {none_rationale}
                    </td>
                </tr>
            </table>
        </div>'''
        display(HTML(html_str))

,Golden Standard,full_system,no_esrs_context
Level,High,High ✓,High ✓
Rationale,"Despite the absence of an ESRS-compliant transition plan, the disclosure quality is High because Aker BP transparently and comprehensively addresses both parts of the question: (1) it explicitly states it does NOT have a transition plan per ESRS requirements due to absence of Scope 3 targets, and (2) it clearly states its existing targets are not science-based. The candid acknowledgment of limitations, combined with detailed disclosure of existing Scope 1&2 measures and a 2017 baseline, cons…","The company's report explicitly discloses that it does not have Scope 3 targets and therefore ‘‘do not have a climate transition plan in line with the ESRS' requirements’’ and that it does not have a timeline for developing one (Evidence 1). This is a clear and unambiguous answer to the question whether a transition plan exists and, if not, the timeline for adoption.","The report explicitly declares (section 2.2) that Aker BP ""does not have any targets to reduce our scope 3 emissions. As a result, we do not have a climate transition plan in line with the ESRS' requirements and do not have a timeline for if or when we will develop such a plan."" This is a clear, direct disclosure that fully answers whether an ESRS-aligned transition plan exists and whether there is a timeline for adopting one. The report separately discloses scope 1/2 decarbonisat…"


,Golden Standard,full_system,no_esrs_context
Level,Medium,Medium ✓,Medium ✓
Rationale,Galp provides partial disclosure: a transition plan is being developed but not yet published. There is no explicit 1.5°C alignment statement. The company takes a 'progressive and pragmatic approach' and is reassessing targets. Capex allocation to low-carbon activities is disclosed but the overall plan lacks the completeness required for High fulfillment.,"Galp clearly discloses that it is developing/maturing an energy transition plan and gives substantive supporting information (e.g., 2024 oil & gas capex, planned low‑carbon capex share, examples of mitigation projects and that targets are being reassessed). However, it has not published a formal transition plan and does not present the ESRS-required elements showing compatibility with a 1.5°C pathway (explicit 1.5°C benchmarking, firm GHG reduction targets aligned to 1.5°C, a comprehensive m…","The retrieved evidence clearly states that a transition plan is being developed and explains the conditions for publication (i.e., after maturing the portfolio assessment), which answers the 'existence' and 'timeline for adoption' parts of the question. However, the report does not provide a published transition plan, nor does it explicitly declare that any plan or targets are aligned with the 1.5°C objective or provide concrete 1.5°C-consistent pathways, dates or interim tar…"


,Golden Standard,full_system,no_esrs_context
Level,Medium,Medium ✓,High ✗
Rationale,"Neste provides partial disclosure: it explicitly states it does NOT have a formal ESRS-compliant transition plan, though it has targets, actions, and resource plans that support climate mitigation. The company demonstrates 1.5°C alignment for Scope 1&2 targets using SBTi, TPI, and ERI frameworks. Fulfillment is Medium because the absence of a formal, dedicated transition plan with all ESRS-required components (governance, implementation roadmap, monitoring) represents an incomplete disclosur…","Neste explicitly states it does not have a separate ESRS-style transition plan (so the specific ESRS disclosure requirement for a distinct transition plan is not fully met). However, the company discloses substantial transition-relevant elements (time-bound GHG targets, stated 1.5°C alignment for Scope 1 & 2, mitigation actions and levers, internal carbon price, Board governance, progress to date and taxonomy/CapEx reporting). These elements partially satisfy the intent of E1-1, but the abse…","The retrieved evidence clearly and substantively answers the question: Neste explicitly states it does not have a separate transition plan, describes the substitute elements it uses (targets, actions, resource plans, internal carbon price, scenario analysis and Board governance) and states it will evaluate developing a specific transition plan. The company also documents that its scenario analysis includes a 1.5°C (Net Zero 2050) pathway and that it considers its scope 1 & 2 2030 target to b…"


,Golden Standard,full_system,no_esrs_context
Level,High,High ✓,High ✓
Rationale,"Aker BP provides comprehensive disclosure of both policies (Code of Conduct, Human Rights Policy) and engagement processes (Business Partner Integrity Procedure, Supplier Declaration, human rights audits with worker interviews, Integrity Channel, performance review meetings). The report includes concrete details on 7 on-site audits across 10 sites in 7 countries covering 400+ workers, with dedicated focus on child/forced labor. Updated IDD processes and contractual obligations demonstrate system…","The retrieved report evidence clearly and substantially describes (1) explicit policy commitments covering forced labour, trafficking and child labour (code of conduct and human rights policy aligned with UNGPs/ILO/OECD), (2) mandatory supplier requirements (supplier declaration and contractual clauses, integrity due diligence), and (3) concrete engagement and remediation processes (risk‑based human rights due diligence cycle, on‑site audits with worker interviews, corrective action plans, follo…","The retrieved report evidence provides clear, substantive disclosures of relevant policies (Code of Conduct, Human Rights Policy), pre-contract screening (business partner integrity procedure, IDD, supplier declaration and contractual clauses), a documented human rights due diligence cycle, targeted audit programmes (on-site and desktop) that explicitly include child and forced labour among audit topics, worker interviews, grievance channels (integrity channel) and corrective action follow-up. T…"


,Golden Standard,full_system,no_esrs_context
Level,High,High ✓,High ✓
Rationale,"Galp provides comprehensive coverage of both policies (four distinct policy frameworks with explicit human rights commitments) and engagement processes (dedicated ESG procurement process, 227 supplier audits explicitly covering child and forced labour, third-party audits with direct worker interaction, grievance mechanisms via OpenTalk and Supply4Galp). Concrete metrics demonstrate systematic implementation with 91% of Tier 1 suppliers assessed.","The retrieved report text clearly and substantially describes (1) the policies that cover value‑chain workers and their alignment with internationally recognised instruments (UNGPs, ILO, OECD), (2) that suppliers are required to follow a Sustainable Procurement Policy and are subject to ESG risk assessments and audits, (3) the specific inclusion of child and forced labour topics in supplier audits, and (4) the engagement and remediation mechanisms (audits, site visits, CAP/IAP, OpenTalk, Supply4…","The retrieved report evidence clearly and substantially describes (1) the relevant policies that govern value‑chain worker rights (including explicit reference to ILO core conventions and UNGP alignment), (2) concrete supplier engagement and identification mechanisms (risk assessments, Counterparty Integrity Verification, Supply4Galp, site visits, and a quantified audit programme), and (3) remediation channels (CAP/IAP, ESMPs, Stop Work Authority) and grievance/reporting channels (OpenTalk). The…"


...

,Golden Standard,full_system,no_esrs_context
Level,High,High ✓,High ✓
Rationale,"Galp provides detailed disclosure of anti-corruption systems (Code of Ethics, Anti-Corruption Policy, KYC process, OpenTalk channel, training programs) and transparent incident reporting (zero incidents, convictions, and fines in 2024), fully addressing both G1-3 and G1-4.","The retrieved report text provides clear, specific information addressing the ESRS G1 requirements: it describes the procedures and tools to prevent/detect/respond (policies, internal controls, KYC, risk assessments, whistleblowing channel), identifies the investigating body and its independence, states that outcomes are escalated to the Audit Board, gives training programme existence and coverage (numbers and % for functions-at-risk), and discloses metrics on incidents/convictions/fines (all ze…","The retrieved report provides clear, specific descriptions of the prevention and detection systems (policies, control manual, risk assessments, KYC/counterparty integrity checks, whistleblowing channel, investigation body, training and supplier platforms) and quantitative metrics (number of counterparties assessed, training coverage). It also explicitly reports the outcomes requested: convictions = 0, confirmed incidents = 0, fines = €0. These items directly and substantially answer the question…"


,Golden Standard,full_system,no_esrs_context
Level,Medium,High ✗,High ✗
Rationale,"Neste demonstrates comprehensive G1-3 anti-corruption systems including Code of Conduct with zero tolerance, Anti-Corruption Principle aligned with UN Convention, Ethics Online reporting channel, Investigation Group, and mandatory training. However, G1-4 (fines for violation of anti-corruption laws and standards of anti-corruption) is explicitly marked as 'Not material' in Neste's ESRS disclosure matrix, meaning the company does not report specific incident counts, convictions, or…","The retrieved report text provides clear, specific descriptions of the anti‑corruption policies, prevention and detection procedures, training approaches, supplier controls, reporting channels and investigation processes, and describes governance and escalation paths. The report explicitly states that no violations were identified with the minimum safeguards (which cover bribery/corruption) and contains no disclosures of convictions, fines or legal cases for corruption — together these substanti…","The retrieved evidence clearly and substantively describes the anti-corruption and anti-bribery systems (policies, training, screening, reporting channels, investigation procedures, governance and supplier measures). The report also explicitly states that no violations of the minimum safeguards (including bribery and corruption) were identified, and the ESRS mapping treats fines/anti-corruption disclosures as not material/not applicable, supporting the conclusion that no confirmed corruption inc…"


,Golden Standard,full_system,no_esrs_context
Level,High,High ✓,High ✓
Rationale,"Aker BP provides a detailed and structured disclosure of its double materiality assessment process, including the methodology (four-step process), stakeholder engagement, assessment criteria for both impact and financial materiality, a clear materiality matrix, identification of all material topics with their relative positioning, and governance oversight. Topic-specific DMA processes are also described for each ESRS environmental standard.","The retrieved report text describes the DMA methodology (four‑step process, top‑down/bottom‑up approaches, stakeholder engagement, use of a standardized severity×likelihood and consequence×probability matrix integrated with enterprise risk management, board/management validation) and explicitly lists the topics classified as high on both impact and financial materiality (climate change, own workforce, pollution, business conduct and resource use & circular economy). It also documents topic‑s…","The retrieved report text provides a clear, substantive description of how the DMA was conducted (four‑step process, top‑down/bottom‑up identification, stakeholder engagement, use of severity×likelihood and alignment with the company risk matrix/ISO 31000, and governance validation). The report also explicitly lists which topics are considered both impact‑ and financially‑material (climate change; own workforce; pollution; business conduct; resource use and circular economy). These elements dire…"


,Golden Standard,full_system,no_esrs_context
Level,High,High ✓,High ✓
Rationale,"Updated to align the answer with the materiality matrix: only climate change, pollution, water and marine resources, and health and safety are material from both perspectives.","The retrieved report text explicitly describes the double materiality process (six-step methodology, frameworks used, stakeholder survey, scoring approach for both impact and financial materiality, consolidation and approval) and provides a materiality table that marks which topics are material for impact and financial perspectives. These elements directly address the ESRS IRO-1 and SBM-3 requirements on describing the process and stating the material matters material under both lenses.","The retrieved report excerpts provide a clear, step-by-step description of the methodology (six-step process, bottom-up approach, criteria/scoring methods for impact and financial materiality, stakeholder inputs, governance approval) and include a materiality matrix/table showing which topics are material for impact and financial perspectives. These elements directly answer both parts of the question with sufficient detail."


,Golden Standard,full_system,no_esrs_context
Level,High,Medium ✗,Medium ✗
Rationale,"Neste provides comprehensive description of its four-phase DMA methodology aligned with ESRS 1, covering value chain mapping, impact and financial assessment, and scoring with thresholds. Material topics are identified from both impact and financial perspectives across environmental, social, and governance dimensions.","The sustainability statement clearly and substantially describes the DMA methodology (four phases, scoring, thresholds, validation, integration with ERM, stakeholder engagement and time‑horizon alignment) — supporting High disclosure fulfillment for the process. However, the evidence does not unambiguously list which specific sustainability matters exceeded materiality thresholds on both the impact and the financial sides. While climate change is explicitly documented as material on both dimensi…","Neste provides a detailed and ESRS‑aligned description of how the double materiality assessment was conducted (value‑chain mapping; impact and financial assessments; scoring, thresholds and validation; stakeholder engagement; Board review) and publishes the set of material topical areas included in the Sustainability statement. This satisfies the process description requirement well. However, the report does not explicitly identify, on a topic‑by‑topic basis, which sustainability matters were de…"


### 9.2 Quantitative Comparison Summary

In [12]:
# Build a comprehensive summary table across all examples
tbl = '''
<table style="border-collapse:collapse; width:100%; font-size:0.88em; margin-top:1em;">
<tr style="background:#2c3e50; color:white;">
    <th style="padding:8px 12px; text-align:left;">Question</th>
    <th style="padding:8px 12px; text-align:left;">Company</th>
    <th style="padding:8px 12px; text-align:center;">Gold Level</th>
    <th style="padding:8px 12px; text-align:center;">full_system</th>
    <th style="padding:8px 12px; text-align:center;">no_esrs_context</th>
    <th style="padding:8px 12px; text-align:center;">Answer Len (full)</th>
    <th style="padding:8px 12px; text-align:center;">Answer Len (no ctx)</th>
</tr>
'''

row_idx = 0
matches_full = 0
matches_none = 0
total_count = 0

for qid in EXAMPLE_QUESTIONS:
    first = examples[qid][EXAMPLE_COMPANIES[0]]
    for comp in EXAMPLE_COMPANIES:
        s = examples[qid][comp]
        r_full = results[(qid, comp, "full_system")]
        r_none = results[(qid, comp, "no_esrs_context")]
        gold = s.fulfillment_level

        full_match = r_full["compliance_level"] == gold
        none_match = r_none["compliance_level"] == gold
        if full_match: matches_full += 1
        if none_match: matches_none += 1
        total_count += 1

        def level_cell(level, match):
            color = {"High": "#27ae60", "Medium": "#f39c12", "Low": "#e74c3c"}.get(level, "#333")
            icon = "✓" if match else "✗"
            icon_color = "#27ae60" if match else "#e74c3c"
            return f'<span style="color:{color}; font-weight:bold;">{level}</span> <span style="color:{icon_color};">{icon}</span>'

        bg = "#f8f9fa" if row_idx % 2 == 0 else "#fff"
        tbl += f'''<tr style="background:{bg};">
            <td style="padding:6px 12px; border-bottom:1px solid #eee;">{qid} ({first.esrs_standard})</td>
            <td style="padding:6px 12px; border-bottom:1px solid #eee;">{comp}</td>
            <td style="padding:6px 12px; text-align:center; border-bottom:1px solid #eee; font-weight:bold;">{gold}</td>
            <td style="padding:6px 12px; text-align:center; border-bottom:1px solid #eee;">{level_cell(r_full["compliance_level"], full_match)}</td>
            <td style="padding:6px 12px; text-align:center; border-bottom:1px solid #eee;">{level_cell(r_none["compliance_level"], none_match)}</td>
            <td style="padding:6px 12px; text-align:center; border-bottom:1px solid #eee;">{len(r_full.get("answer", "")):,}</td>
            <td style="padding:6px 12px; text-align:center; border-bottom:1px solid #eee;">{len(r_none.get("answer", "")):,}</td>
        </tr>'''
        row_idx += 1

# Summary row
tbl += f'''<tr style="background:#2c3e50; color:white; font-weight:bold;">
    <td colspan="3" style="padding:8px 12px;">Accuracy</td>
    <td style="padding:8px 12px; text-align:center;">{matches_full}/{total_count} ({100*matches_full/total_count:.0f}%)</td>
    <td style="padding:8px 12px; text-align:center;">{matches_none}/{total_count} ({100*matches_none/total_count:.0f}%)</td>
    <td colspan="2"></td>
</tr>'''
tbl += '</table>'

display(HTML(tbl))

Question,Company,Gold Level,full_system,no_esrs_context,Answer Len (full),Answer Len (no ctx)
Q3 (ESRS E1),Aker BP,High,High ✓,High ✓,228,398
Q3 (ESRS E1),Galp,Medium,Medium ✓,Medium ✓,550,496
Q3 (ESRS E1),Neste,Medium,Medium ✓,High ✗,638,655
Q9 (ESRS S2),Aker BP,High,High ✓,High ✓,"1,656","2,107"
Q9 (ESRS S2),Galp,High,High ✓,High ✓,"1,170","1,359"
Q9 (ESRS S2),Neste,High,High ✓,High ✓,"1,500","2,091"
Q12 (ESRS G1),Aker BP,High,High ✓,High ✓,962,"1,125"
Q12 (ESRS G1),Galp,High,High ✓,High ✓,799,985
Q12 (ESRS G1),Neste,Medium,High ✗,High ✗,"1,117","1,183"
Q1 (ESRS 2),Aker BP,High,High ✓,High ✓,"1,143","1,680"


## 10. Qualitative Observations

Looking across these examples spanning four ESRS standards (E1, S2, G1, ESRS 2) and three
companies (Aker BP, Galp, Neste), several patterns emerge that are representative of the
broader RQ2 findings:

**Compliance level accuracy:** In our 12-sample subset, `full_system` matches the gold standard
in 10 out of 12 cases (83%), while `no_esrs_context` matches in 9 out of 12 (75%). This
marginal advantage for context injection is consistent with the aggregate finding that ESRS
context provides a slight — but not dramatic — improvement in level prediction.

**Where context injection helps:**
- For **ESRS E1 × Neste** (transition plan, gold: Medium), `full_system` correctly predicts
  Medium while `no_esrs_context` over-predicts High. The detailed regulatory requirements
  (GHG targets, CapEx alignment, Paris Agreement compatibility) likely helped the model
  identify specific gaps that warranted a Medium rather than High rating.

**Where both conditions struggle equally:**
- For **ESRS G1 × Neste** (anti-corruption, gold: Medium), both conditions over-predict High.
  This suggests that the evidence was sufficiently convincing on the surface, and neither the
  presence nor absence of regulatory context corrected the model's assessment.
- For **ESRS 2 × Neste** (general disclosures, gold: High), both conditions under-predict
  Medium. Here the model appears to have been overly conservative regardless of context.

**Rationale style differences:**
- The `full_system` rationale tends to explicitly reference ESRS disclosure requirements
  (e.g., "E1-1", "G1-3") when explaining why a compliance level was assigned. The injected
  regulatory text provides a structural framework for mapping company disclosures against
  expected elements.
- The `no_esrs_context` rationale relies entirely on the content of the retrieved evidence,
  using more general compliance reasoning. It tends to focus more on what the evidence
  contains rather than what the standard requires.

**Standard-specific observations:**
- **ESRS E1 (Transition Plan):** The regulatory context is particularly detailed (GHG targets,
  decarbonization levers, CapEx alignment). When injected, the model uses these criteria to
  structure its assessment — and this is the one standard where context injection demonstrably
  improved accuracy in our examples.
- **ESRS G1 (Business Conduct):** Anti-corruption and training requirements are relatively
  specific. Both conditions perform similarly since the disclosure requirements align well
  with common governance expectations.
- **ESRS S2 (Workers in Value Chain):** Social standards involve nuanced policy and engagement
  assessments. Both conditions achieve perfect accuracy on our three companies, suggesting
  the retrieved evidence was sufficiently clear.
- **ESRS 2 (General Disclosures):** As the most cross-cutting standard, the model's assessment
  can vary — both conditions struggled equally with Neste's double materiality disclosure.

**Overall pattern:** These results align with the aggregate RQ2 finding: ESRS context injection
provides a modest improvement in compliance-level prediction accuracy, primarily for standards
with highly specific disclosure requirements (like E1). The evidence retrieved by the RQ1
pipeline is generally sufficient for the LLM to reason about disclosure fulfillment. The primary
effect of the injection is on rationale style (more structured, ESRS-referenced language) and
on edge cases where regulatory specificity matters.

## 11. Appendix — Full Raw Responses

For completeness, the full JSON responses from both conditions for all example combinations
are shown below. Each entry can be expanded to inspect the complete model output.

In [13]:
# Show full raw responses as expandable HTML details blocks
for qid in EXAMPLE_QUESTIONS:
    first = examples[qid][EXAMPLE_COMPANIES[0]]
    display(HTML(f'<h4 style="margin-top:1.5em;">{qid} — {first.esrs_standard}</h4>'))

    for comp in EXAMPLE_COMPANIES:
        for condition in ["full_system", "no_esrs_context"]:
            result = results[(qid, comp, condition)]
            display_result = {k: v for k, v in result.items() if k != "raw_response"}
            cond_label = "WITH ESRS" if condition == "full_system" else "WITHOUT ESRS"
            color = "#3498db" if condition == "full_system" else "#e74c3c"

            json_str = json.dumps(display_result, indent=2, ensure_ascii=False)
            display(HTML(f'''
            <details style="margin:4px 0;">
                <summary style="cursor:pointer; padding:6px 10px; background:#f8f9fa; border:1px solid #ddd;
                               border-radius:4px; font-size:0.9em;">
                    <strong>{html_mod.escape(comp)}</strong> — <span style="color:{color};">{cond_label}</span>
                    → <strong>{result["compliance_level"]}</strong>
                </summary>
                <pre style="font-size:0.8em; background:#fff; padding:10px; border:1px solid #eee;
                            margin:4px 0; max-height:300px; overflow-y:auto;">{html_mod.escape(json_str)}</pre>
            </details>'''))